<a href="https://colab.research.google.com/github/sruthi-analyst/Messy-Medical-ETL/blob/main/Medical_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [169]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [170]:
raw_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/realworld_medical_dirty.csv")
print("Successfull read csv data")
print(f"\nShape: {raw_df.shape[0]} rows and {raw_df.shape[1]} columns")
print(f"\nColumns: {raw_df.columns.to_list()}")

Successfull read csv data

Shape: 100 rows and 10 columns

Columns: ['Patient_ID', 'Age', 'Gender', 'Blood_Pressure', 'Cholesterol', 'BMI', 'Smoker', 'Diagnosis', 'Admission_Date', 'Notes']


In [171]:
print("\n",'=='*35,"\n")
print("\t\t    DATA QUALITY DOAGNOSIS REPORT")
print("\n",'=='*35,"\n")

print("MISSING VALUES")
print("\n",'*~'*35,"\n")
print("COLUMN-WISE NULL VALUES: \n")
print(raw_df.isna().sum())
print("total No. of NULL Values: ", raw_df.isna().sum().sum())
print("\n",'*~'*35,"\n")

print("ROW-WISE NULL VALUES: \n")
print(raw_df.isna().sum(axis=1).head(8))
print("\n",'*~'*35,"\n")

imp_cols = ['Age', 'Gender', 'Blood_Pressure', 'Cholesterol', 'BMI', 'Smoker']
remove_rows = raw_df[raw_df[imp_cols].isna().sum(axis=1)>=3]
print(f"No. of Rows missing MORE THAN 2 IMPORTANT ATTRIBUTES: {remove_rows['Patient_ID'].count()}")
print(remove_rows[imp_cols])
print(f"\nIndices of rows to be removed: {remove_rows.index.to_list()}")
print("\n",'*~'*35,"\n")

remove_rows2 = raw_df[raw_df[imp_cols].isna().sum(axis=1)>=2]
print(f"No. of Rows missing MORE THAN 1 IMPORTANT ATTRIBUTES: {remove_rows2['Patient_ID'].count()}")
print(remove_rows2[imp_cols])
print(f"\nIndices of rows to be removed: {remove_rows2.index.to_list()}")
print("\n",'*~'*35,"\n")

print("DUPLICATE VALUES: ", raw_df.duplicated().sum())
print("\n",'*~'*35,"\n")

print("DATA TYPE\n")
print(raw_df.dtypes)
print("\n",'*~'*35,"\n")

print("Unique Diseases Diagnosed: ", raw_df['Diagnosis'].unique())
print("\n",'*~'*35,"\n")



		    DATA QUALITY DOAGNOSIS REPORT


MISSING VALUES

 *~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~ 

COLUMN-WISE NULL VALUES: 

Patient_ID         0
Age               17
Gender            15
Blood_Pressure    14
Cholesterol       20
BMI               20
Smoker            20
Diagnosis         48
Admission_Date     0
Notes             43
dtype: int64
total No. of NULL Values:  197

 *~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~ 

ROW-WISE NULL VALUES: 

0    2
1    3
2    2
3    3
4    2
5    0
6    3
7    2
dtype: int64

 *~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~*~ 

No. of Rows missing MORE THAN 2 IMPORTANT ATTRIBUTES: 6
     Age  Gender  Blood_Pressure  Cholesterol   BMI Smoker
27  25.0    MALE           160.0          NaN   NaN    NaN
30  45.0    MALE             NaN          NaN  35.4    NaN
39  55.0    Male             NaN          NaN   NaN    NaN
69   NaN  Female           130.0        250.0   NaN   

PHASE-1
*   Drop row 92
*   Among Age,Gender,Blood_Pressure,Cholesterol, BMI,Smoker number of null values>=2 - drop rows
* Handled all NULL values
* Text formatting for 'Gender'

PHASE-2
* Type Casting 'Age'
* BP Categorisation - low, normal, high (based on range)
* Categorise Fitness - obese, normal, underweight (based on BMI, Age and Gender)
* Randomly generate 'hospitalised days'
* Calculate discharge date (using admission date & hospitalised days)

PHASE-3
* Possible diagnosis prediction - like "possible" heart attack, cancer, chronic illness (using BMI, Cholestrol, BP, Smoker, age)
* Add follow ups for chronic diagnosis like possible chronic illness (longer stay and follow up notes)

In [193]:
#Creating a working copy
df = raw_df.copy()
imputations = {}
print("Working Copy Created")
print('*~'*10)

#Removing rows that contribute less data
df.drop(remove_rows2.index, inplace=True)
print("\nRemoved rows with more than 1 missing important attributes")
print(f"No. of rows removed: {raw_df.shape[0]-df.shape[0]}\nNew no. of rows: {df.shape[0]}")

#Null value handling - Diagnosis and Notes Column
imputations['Diagnosis'] = df['Diagnosis'].isna().sum()
df['Diagnosis'].fillna('Undiagnosed', inplace=True)

imputations['Notes'] = df['Notes'].isna().sum()
df['Notes'].fillna('Eluthanum', inplace=True)
print("\nTemporarily replaced Null values in 'Diagnosis' and 'Notes' column for readability\n")

print('*~'*10)

print("Total nulls remaining: ", df.isna().sum().sum())
print(f"Number of nulls handled: {raw_df.isna().sum().sum()-df.isna().sum().sum()}")


Working Copy Created
*~*~*~*~*~*~*~*~*~*~

Removed rows with more than 1 missing important attributes
No. of rows removed: 24
New no. of rows: 76

Temporarily replaced Null values in 'Diagnosis' and 'Notes' column for readability

*~*~*~*~*~*~*~*~*~*~
Total nulls remaining:  51
Number of nulls handled: 146


In [194]:
#AGE
#Null value handling - Age Column
heart_median = df[df['Diagnosis'] == 'Heart Disease']['Age'].median()
heart_disease_age_null = df[df['Diagnosis'] == 'Heart Disease']['Age'].isna().sum()
df.loc[df['Diagnosis'] == 'Heart Disease', 'Age'] = df.loc[df['Diagnosis'] == 'Heart Disease', 'Age'].fillna(heart_median)

diabetes_median = df[df['Diagnosis'] == 'Diabetes']['Age'].median()
diabetes_age_null = df[df['Diagnosis'] == 'Diabetes']['Age'].isna().sum()
df.loc[df['Diagnosis'] == 'Diabetes', 'Age'] = df.loc[df['Diagnosis'] == 'Diabetes', 'Age'].fillna(diabetes_median)

imputations['Age'] = heart_disease_age_null + diabetes_age_null
print("\nNull values in Age column imputed by median according to diagnosed disease successfully!")

#Dropping remaining 4 NaN Age null rows
df.dropna(subset=['Age'], inplace=True)
print("Dropped remaining 4 rows with Null values for Age successfully!\n")

print('*~'*10)

print("Total nulls remaining: ", df.isna().sum().sum())
print(f"Number of nulls handled: {raw_df.isna().sum().sum()-df.isna().sum().sum()}")



Null values in Age column imputed by median according to diagnosed disease successfully!
Dropped remaining 4 rows with Null values for Age successfully!

*~*~*~*~*~*~*~*~*~*~
Total nulls remaining:  40
Number of nulls handled: 157


In [195]:
#GENDER
#Text Formatting - Gender Column
df['Gender'] = df['Gender'].str.strip().str.title()
print("\nFormatted Gender legibly!")

#Null value handling - Gender Column - imputing them with proportion of Male:Female
null_gender_indices = df[df['Gender'].isnull()].index
null_gender_sum = df['Gender'].isnull().sum()
imputations['Gender'] = null_gender_sum

#calculate original probabilities of gender categories
probabilities = df['Gender'].dropna().value_counts(normalize=True)  #values_counts() returns frequency table without normalise=True, with it it returns the probability table

#Generate N Randomly positioned gender categories mimicing the original probability calculated, where n is the number of nulls in gender to be handled.
random_genders = np.random.choice(probabilities.index, size=null_gender_sum, p=probabilities.values)

#Fill the null values
df.loc[null_gender_indices, 'Gender'] = random_genders
print("Gender null values imputed based on proportion successfully!\n")

print('*~'*10)

print("Total nulls remaining: ", df.isna().sum().sum())
print(f"Number of nulls handled: {raw_df.isna().sum().sum()-df.isna().sum().sum()}")



Formatted Gender legibly!
Gender null values imputed based on proportion successfully!

*~*~*~*~*~*~*~*~*~*~
Total nulls remaining:  30
Number of nulls handled: 167


In [196]:
#SMOKER
#Null value handling - Smoker column
imputations['Smoker'] = df['Smoker'].isna().sum()
df['Smoker'].fillna('Unknown', inplace=True)
print("\nSmoker null values imputed with 'Unknown' successfully!\n")

print('*~'*10)

print("Total nulls remaining: ", df.isna().sum().sum())
print(f"Number of nulls handled: {raw_df.isna().sum().sum()-df.isna().sum().sum()}")



Smoker null values imputed with 'Unknown' successfully!

*~*~*~*~*~*~*~*~*~*~
Total nulls remaining:  21
Number of nulls handled: 176


In [197]:
#BP, Cholesterol and BMI
imputations['BMI'] = df['BMI'].isna().sum()
imputations['BP'] = df['Blood_Pressure'].isna().sum()
imputations['Cholesterol'] = df['Cholesterol'].isna().sum()

#Null value handling - BMI, BP, Cholesterol
#Multiple Bayesian Regression Imputation
impute_cols = ['Blood_Pressure', 'BMI', 'Cholesterol']
imputer = IterativeImputer(BayesianRidge(), max_iter=10, random_state=0)
df_copy = df[impute_cols].copy()
df[impute_cols] = imputer.fit_transform(df_copy)
print("\nBMI, BP and Cholesterol null values imputed using Multivariate Bayesian Regression Imputer successfully!\n")

print('*~'*10)

print("Total nulls remaining: ", df.isna().sum().sum())
print(f"Number of nulls handled: {raw_df.isna().sum().sum()-df.isna().sum().sum()}")



BMI, BP and Cholesterol null values imputed using Multivariate Bayesian Regression Imputer successfully!

*~*~*~*~*~*~*~*~*~*~
Total nulls remaining:  0
Number of nulls handled: 197


In [198]:
#Printing Imputations
nulls = 0
for key in imputations:
  print(key,":", imputations[key])
  nulls+=imputations[key]
print("\nTotal nulls imputed: ", nulls)

Diagnosis : 37
Notes : 31
Age : 9
Gender : 10
Smoker : 9
BMI : 6
BP : 7
Cholesterol : 8

Total nulls imputed:  117


PHASE-2
* Text Formatting Smoker
* Type Casting 'Age'
* BP Categorisation - low, normal, high (based on range)
* Categorise Fitness - obese, normal, underweight (based on BMI, Age and Gender)
* Randomly generate 'hospitalised days'
* Calculate discharge date (using admission date & hospitalised days)
* Round decimals to 2 places in Cholesterol and BMI
* Reorder columns for readibility

In [199]:
#Analysing and formatting Smoker
print(df['Smoker'].unique())

df['Smoker'].replace('Yes', 'Y', inplace=True)
df['Smoker'].replace('No', 'N', inplace=True)
print("Replaced all yes denoting values with 'Y' and no denoting values with 'N'")

print(df['Smoker'].unique())

['Yes' 'Unknown' 'No' 'N' 'Y']
Replaced all yes denoting values with 'Y' and no denoting values with 'N'
['Y' 'Unknown' 'N']


In [200]:
#Type Cast 'Age'
df['Age'] = df['Age'].astype(int)
print("Age type casted:", df['Age'].dtype)

#ROUND BMI to 2 DECIMAL PLACES
df['BMI'] = df['BMI'].round(1)
print("Rounded BMI to 2 decimal places")
#ROUND CHOLESTEROL TO # DECIMAL PLACES
df['Cholesterol'] = df['Cholesterol'].round(1)
print("Rounded Cholesterol to rounded to 2 decimal places")

Age type casted: int64
Rounded BMI to 2 decimal places
Rounded Cholesterol to rounded to 2 decimal places


In [201]:
# BP Categorization
# LOW < 90
# 90<= NORMAL <=120
# HIGH > 120

#Typecasting BP to INT
df['Blood_Pressure'] = df['Blood_Pressure'].astype(int)

#Categorising BP
Bins = [0, 89, 120, 299, 369, np.inf]
Labels = ['Low', 'Normal', 'High', 'Miraculous', 'Dead']
df['BP_Level'] = pd.cut(df['Blood_Pressure'], bins=Bins, labels=Labels)
print(df[['Blood_Pressure', 'BP_Level']].head())
print("Categorised BP Levels successfully!")

   Blood_Pressure BP_Level
0             120   Normal
1             150     High
2             120   Normal
3             139     High
4             120   Normal
Categorised BP Levels successfully!


In [202]:
# Fitness Calculation - BMI Categorization
# Senior Citizen (Age>=60) (BMI<22) -> UNDERWEIGHT
# BMI[22, 26.9] -> NORMAL; BMI[27, 29.9] -> OVERWEIGHT; BMI[30, any] -> OBESE
# Adult (20<Age<60) (BMI<18.5) -> UNDERWEIGHT
# BMI[18.5, 24.9] -> NORMAL; BMI[25.0, 29.9] -> OVERWEIGHT; BMI[30.0, any] -> OBESE
# Toddler (2<Age<5) (BMI<13.5) -> UNDERWEIGHT
# BMI[13.5, 16.8] -> NORMAL; BMI[16.8, 18.5] -> OVERWEIGHT; BMI[18.5, any] -> OBESE
# FEMALE
# Teens (14<Age<19) (BMI<16.5) -> UNDERWEIGHT
# BMI[16.5, 25] -> NORMAL; BMI[25, 29] -> OVERWEIGHT; BMI[29, any] -> OBESE
# Pre-Teens (10<Age<13) (BMI<14.5) -> UNDERWEIGHT
# BMI[15, 22] -> NORMAL; BMI[22,25] -> OVERWEIGHT; BMI[25, any]->OBESE
# Childhood (5<Age<9) (BMI<14) -> UNDERWEIGHT
# BMI[14, 18.5] -> NORMAL; BMI[18.5, 21.0] -> OVERWEIGHT; BMI[21.0, any] -> OBESE
# MALE
# Teens (14<Age<19) (BMI<16) -> UNDERWEIGHT
# BMI[16, 24] -> NORMAL; BMI[24, 27] -> OVERWEIGHT; BMI[27, any] -> OBESE
# Pre-Teens (10<Age<13) (BMI<14.5) -> UNDERWEIGHT
# BMI[15, 21] -> NORMAL; BMI[21,24] -> OVERWEIGHT; BMI[24, any]->OBESE
# Childhood (5<Age<9) (BMI<14) -> UNDERWEIGHT
# BMI[14, 18.0] -> NORMAL; BMI[18.0, 20.0] -> OVERWEIGHT; BMI[20.0, any] -> OBESE

def fit_calc(age, bmi, gender):
  if age>=60:
    if bmi<22:
      return 'Underweight'
    elif 22<=bmi<=26.9:
      return 'Normal'
    elif 27<=bmi<=29.9:
      return 'Overweight'
    else:
      return 'Obese'
  elif 20<=age<60:
    if bmi<18.5:
      return 'Underweight'
    elif 18.5<=bmi<=24.9:
      return 'Normal'
    elif 25<=bmi<=29.9:
      return 'Overweight'
    else:
      return 'Obese'
  elif 2<=age<=4:
    if bmi<13.5:
      return 'Underweight'
    elif 13.5<=bmi<=16.8:
      return 'Normal'
    elif 16.8<=bmi<=18.5:
      return 'Overweight'
    else:
      return 'Obese'
  elif gender == 'Female':
    if 14<=age<=19:
      if bmi<16.5:
        return 'Underweight'
      elif 16.5<=bmi<=25:
        return 'Normal'
      elif 25<=bmi<=29:
        return 'Overweight'
      else:
        return 'Obese'
    elif 10<=age<=13:
      if bmi<14.5:
        return 'Underweight'
      elif 14.5<=bmi<=22:
        return 'Normal'
      elif 22<=bmi<25:
        return 'Overweight'
      else:
        return 'Obese'
    elif 5<=age<=9:
      if bmi<14:
        return 'Underweight'
      elif 14<=bmi<=18.5:
        return 'Normal'
      elif 18.5<bmi<21:
        return 'Overweight'
      else:
        return 'Obese'
  else:
    if 14<=age<=19:
      if bmi<16:
        return 'Underweight'
      elif 16<=bmi<=24:
        return 'Normal'
      elif 24<bmi<=27:
        return 'Overweight'
      else:
        return 'Obese'
    elif 10<=age<=13:
      if bmi<14.5:
        return 'Underweight'
      elif 14.5<=bmi<=21:
        return 'Normal'
      elif 21<bmi<24:
        return 'Overweight'
      else:
        return 'Obese'
    elif 5<=age<=9:
      if bmi<14:
        return 'Underweight'
      elif 14<=bmi<=18:
        return 'Normal'
      elif 18<bmi<20:
        return 'Overweight'
      else:
        return 'Obese'

In [203]:
df["Fitness"] = df.apply(
    lambda row: fit_calc(row["Age"], row["BMI"], row["Gender"]),
    axis=1
)
print("Fitness calculated successfully!")

Fitness calculated successfully!


In [204]:
#Generating Hospitalised Days
def days_stay(diagnosis, age):
  if diagnosis == 'Diabetes':
    return 0
  elif diagnosis == 'Heart_Disease':
    if age<=40:
      return np.random.randint(1, 8)
    else:
      return np.random.randint(5, 14)
  else:
    return np.random.randint(1, 20)

df['Hospitalised_Days'] = df.apply(lambda row: days_stay(row['Diagnosis'], row['Age']), axis=1)
print("Hospitalised Days calculated successfully!")

Hospitalised Days calculated successfully!


In [205]:
#Calculating Discharge Date
df['Admission_Date'] = pd.to_datetime(df['Admission_Date']).dt.date
df['Discharge_Date'] = (pd.to_datetime(df['Admission_Date']) + pd.to_timedelta(df['Hospitalised_Days'], unit='D'))
print("Discharge Date added successfully (Date only format)!")

Discharge Date added successfully (Date only format)!


In [206]:
#REORDERING COLUMNS AS NECESSARY
df = df[['Patient_ID', 'Age', 'Gender', 'Blood_Pressure', 'BP_Level', 'Cholesterol', 'BMI', 'Fitness', 'Smoker', 'Diagnosis', 'Admission_Date', 'Hospitalised_Days', 'Discharge_Date', 'Notes']]

In [209]:
df

,Patient_ID,Age,Gender,Blood_Pressure,BP_Level,Cholesterol,BMI,Fitness,Smoker,Diagnosis,Admission_Date,Hospitalised_Days,Discharge_Date,Notes
0,P1000,55,Female,120,Normal,241.1,35.4,Obese,Y,Heart Disease,2023-12-27,11,2024-01-07,Eluthanum
1,P1001,65,Male,150,High,180.0,27.8,Overweight,Unknown,Undiagnosed,2023-01-01,4,2023-01-05,Eluthanum
2,P1002,45,Male,120,Normal,220.0,22.5,Normal,N,Undiagnosed,2023-12-14,8,2023-12-22,Eluthanum
3,P1003,65,Male,139,High,180.0,35.4,Obese,N,Undiagnosed,2023-07-09,9,2023-07-18,Eluthanum
4,P1004,65,Male,120,Normal,300.0,40.1,Obese,Unknown,Undiagnosed,2023-07-10,1,2023-07-11,Follow-up required
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,P1095,55,Male,150,High,200.0,35.4,Obese,Y,Undiagnosed,2023-07-16,18,2023-08-03,Follow-up required
96,P1096,55,Male,137,High,220.0,27.8,Overweight,Y,Undiagnosed,2023-12-17,1,2023-12-18,Follow-up required
97,P1097,65,Female,150,High,200.0,27.8,Overweight,N,Undiagnosed,2023-05-13,11,2023-05-24,Checkup due
98,P1098,25,Male,139,High,180.0,40.1,Obese,Y,Diabetes,2023-12-24,0,2023-12-24,Follow-up required
